In [ ]:
import mcstasscript as ms
import mcstastox as mx
import scipp as sc
from scipp.typing import VariableLike
import scipp as sc
from scippneutron.conversion.graph.beamline import beamline
from trex_reduction import inelastic
from trex_reduction import produce_trex_event_object
import os

In [ ]:
cwd = os.getcwd()
file_path = cwd + "/isis_let_test"

with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp_simple(source_name="SourceMantid",
                                            sample_name="iso_samp")
    
data = ms.load_data(file_path)

In [ ]:
with mx.Read(file_path) as loaded_data:
    print("=== All components ===")
    print(loaded_data.get_components())
    print("=== Components with data ===")
    print(loaded_data.get_components_with_data())
    print("=== Components with IDs ===")
    print(loaded_data.get_components_with_ids()) 
    print("=== Components with geometry ===")
    print(loaded_data.get_components_with_geometry())

In [ ]:
with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp_simple(source_name="SourceMantid",
                                            sample_name="iso_samp")
    
# Load event data into scipp 
event_object = scipp_object
# McStas provides absolute time, not time of flight
event_object.coords["tof"] = event_object.coords["t"]
# Add additional information required for inelastic scattering
event_object = produce_trex_event_object(event_object, file_path, "Monitor6")
event_object

In [ ]:
qens_graph = {**beamline(scatter=True), **inelastic}
sc.show_graph(qens_graph)

In [ ]:
event_object = event_object.transform_coords("dE", graph=qens_graph)
event_object = event_object.transform_coords("mag_Q", graph=qens_graph)
event_object

In [ ]:
(event_object.hist(dE = 500)).plot()



In [ ]:
event_object.hist(mag_Q = 50).plot()

In [ ]:
event_object.hist(mag_Q = 10, dE = 100).plot()

In [ ]:


pp.slicer(event_object.hist(mag_Q=10, dE=500), keep=['dE'], logc=True)
# fig.children[0].ax.set_xlim(-4, 4)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append("./plet_data")
from plet_data import PletData

plet_data = PletData(cwd+'/plet_data/benzene_290_360_inc.nxspe', 3.6, omega_lims = [-2,2])
fig, ax = plt.subplots()

hist = event_object.hist(dE=500)
dE = hist.coords['dE'].values  # bin edges, length 501
n = hist.values                 # counts, length 500
ax.stairs(n / np.max(n) , dE, label = 'sim')
ax.stairs(plet_data.data.nansum('q').values / np.max(plet_data.data.nansum('q').values), plet_data.data.coords['omega'].values, label = 'pLET-exp')


ax.set_xlabel('dE (meV)')
ax.set_ylabel('Counts')
ax.legend()
plt.show()